In [ ]:
!pip install pageindex

In [ ]:
from pageindex import PageIndexClient
import pageindex.utils as utils

In [ ]:
PAGEINDEX_API_KEY = "Your-pageindex-api-kay"
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [ ]:
import openai
OPENAI_API_KEY = "your-openai-api-key"

In [ ]:
async def call_llm(prompt, model = 'gpt-4o-mini', temperature = 0):
  client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
  response = await client.chat.completions.create(
      model = model,
      messages = [{"role":"user","content":prompt}],
      temperature=temperature
  )
  return response.choices[0].message.content.strip()

In [ ]:
import os, requests
pdf_path = "/content/drive/MyDrive/RAG/docs/ResNet.pdf"

In [ ]:
doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print("Document Submitted:", doc_id)

Document Submitted: pi-cmm53dsfn04e708o9v1wm26v0


Get submitted tree structure

In [ ]:
if pi_client.is_retrieval_ready(doc_id):
  tree = pi_client.get_tree(doc_id, node_summary=True)['result']
  print("Tree Structure")
  utils.print_tree(tree)
else:
  print("Error while processing document")

Tree Structure
[{'title': 'Deep Residual Learning for Image Recogni...',
  'node_id': '0000',
  'prefix_summary': '# Deep Residual Learning for Image Recog...',
  'nodes': [{'title': 'Abstract',
             'node_id': '0001',
             'summary': 'This text introduces a residual learning...'},
            {'title': '1. Introduction',
             'node_id': '0002',
             'summary': 'The text discusses the importance of dep...'},
            {'title': '2. Related Work',
             'node_id': '0003',
             'summary': 'This section reviews related work on res...'},
            {'title': '3 Deep Residual Learning',
             'node_id': '0004',
             'prefix_summary': '## 3 Deep Residual Learning\n',
             'nodes': [{'title': '3.1 Residual Learning',
                        'node_id': '0005',
                        'summary': 'This text introduces residual learning, ...'},
                       {'title': '3.2 Identity Mapping by Shortcuts',
           

Reasoning Based Retrieval with Tree Search

In [ ]:
import json

In [ ]:
query = "Which experiments are there in this document?"

In [ ]:
tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])


In [ ]:
search_prompt = f"""
You are given a question and tree structure of a document.
Each node contains a node id, node title, and corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question : {query}

Document tree structure:
{json.dumps(tree_without_text, indent = 2)}

Plese reply in the following JSON format:
{{
  'thinking':'<Your thinking process>',
  'node_list':['node_id_1', 'node_id_2', ...., 'node_id_n']
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)

Print retrieved nodes and reasoning process

In [ ]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

In [ ]:
print("Reasoning Process:")
utils.print_wrapped(tree_search_result_json['thinking'])

print("\Retrieved Nodes:")
for node_id in tree_search_result_json["node_list"]:
  node = node_map[node_id]
  print(f"Node ID:{node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for experiments mentioned in the document. The most relevant section is '4.
Experiments', which contains subsections detailing various experiments conducted, including those on
ImageNet, CIFAR-10, and object detection. Each of these subsections likely contains specific
information about the experiments performed. Therefore, I will include the node IDs for the
'Experiments' section and its subsections.
\Retrieved Nodes:
Node ID:0009	 Page: 4	 Title: 4. Experiments
Node ID:0010	 Page: 4	 Title: 4.1. ImageNet Classification
Node ID:0011	 Page: 7	 Title: 4.2. CIFAR-10 and Analysis
Node ID:0012	 Page: 8	 Title: 4.3. Object Detection on PASCAL and MS COCO


<>:4: SyntaxWarning: invalid escape sequence '\R'
<>:4: SyntaxWarning: invalid escape sequence '\R'
/tmp/ipython-input-14678/685128508.py:4: SyntaxWarning: invalid escape sequence '\R'
  print("\Retrieved Nodes:")


Answer Generation

In [ ]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]['text'] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

## 4. Experiments


### 4.1. ImageNet Classification

We evaluate our method on the ImageNet 2012 classification dataset [36] that consists of 1000
classes. The models are trained on the 1.28 million training images, and evaluated on the 50k
validation images. We also obtain a final result on the 100k test images, reported by the test
server. We evaluate both top-1 and top-5 error rates.

Plain Networks. We first evaluate 18-layer and 34-layer plain nets. The 34-layer plain net is in
Fig. 3 (middle). The 18-layer plain net is of a similar form. See Table 1 for detailed
architectures.

The results in Table 2 show that the deeper 34-layer plain net has higher validation error than the
shallower 18-layer plain net. To reveal the reasons, in Fig. 4 (left) we compare their
training/validation errors during the training procedure. We have observed the degradation problem -
the

|  layer name | output size | 18-layer | 34-layer | 50-layer | 101-layer | 152-layer  |
| --- |

Generate answer based on retrieved context

In [ ]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print("Generated Answer:\n")
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

The document describes two main sets of experiments:

1. **ImageNet Classification**: This includes evaluations of plain networks and residual networks
(ResNets) of various depths (18, 34, 50, 101, and 152 layers) on the ImageNet 2012 classification
dataset. The experiments focus on comparing top-1 and top-5 error rates, analyzing the degradation
problem in deeper networks, and assessing the effectiveness of identity vs. projection shortcuts.

2. **CIFAR-10 and Analysis**: This section involves experiments on the CIFAR-10 dataset using plain
and residual architectures with varying depths (20, 32, 44, 56, and 110 layers). The focus is on the
behavior of deep networks, optimization difficulties, and the performance of a 1202-layer ResNet.
Additionally, it includes an analysis of layer responses and explores the performance of deep
networks in terms of classification error.

3. **Object Detection on PASCAL and MS COCO**: This part evaluates the performance of ResNet-101